In [0]:
ruta_devoluciones = "/Volumes/electrocasa_dev/bronze/landing/devoluciones/devoluciones.csv"

devoluciones = (
    spark.read
        .option("header", "true")
        .csv(ruta_devoluciones)
)

devoluciones.printSchema()
display(devoluciones.limit(10))

In [0]:
devoluciones.createOrReplaceTempView("devoluciones_tmp")

In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT devolucion_id) AS devoluciones_unicas,

    SUM(CASE
        WHEN CAST(monto_reembolso AS DOUBLE) < 0
        THEN 1 ELSE 0
    END) AS reembolsos_negativos,

    SUM(CASE
        WHEN motivo IS NULL
          OR TRIM(motivo) = ''
        THEN 1 ELSE 0
    END) AS motivo_faltante,

    SUM(CASE
        WHEN pedido_id IS NULL
          OR TRIM(pedido_id) = ''
        THEN 1 ELSE 0
    END) AS pedido_faltante

FROM devoluciones_tmp;

In [0]:
%sql

SELECT
    devolucion_id,
    COUNT(*) AS registros,
    COUNT(DISTINCT concat_ws(
        '||',
        pedido_id,
        sucursal_id,
        producto_id,
        motivo,
        monto_reembolso,
        fecha_devolucion
    )) AS versiones_distintas
FROM devoluciones_tmp
GROUP BY devolucion_id
HAVING COUNT(*) > 1
ORDER BY devolucion_id;

In [0]:
%sql

SELECT
    COUNT(*) AS filas_huerfanas,
    COUNT(DISTINCT d.devolucion_id) AS devoluciones_huerfanas,
    COUNT(DISTINCT d.producto_id) AS productos_huerfanos
FROM devoluciones_tmp d
LEFT ANTI JOIN (
    SELECT DISTINCT producto_id
    FROM electrocasa_dev.silver.productos
) p
ON d.producto_id = p.producto_id;

In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT devolucion_id) AS devoluciones_unicas,
    COUNT(DISTINCT archivo_origen) AS archivos_origen,
    SUM(CASE
        WHEN _rescued_data IS NOT NULL
        THEN 1 ELSE 0
    END) AS registros_rescatados
FROM electrocasa_dev.bronze.devoluciones;

In [0]:
%sql

SELECT
    (SELECT COUNT(*)
     FROM electrocasa_dev.silver.devoluciones) AS devoluciones_validas,

    (SELECT COUNT(*)
     FROM electrocasa_dev.silver.devoluciones_cuarentena) AS devoluciones_cuarentena,

    (
        SELECT COUNT(*)
        FROM (
            SELECT producto_huerfano
            FROM electrocasa_dev.silver.devoluciones

            UNION ALL

            SELECT producto_huerfano
            FROM electrocasa_dev.silver.devoluciones_cuarentena
        )
        WHERE producto_huerfano = true
    ) AS devoluciones_huerfanas,

    (
        SELECT COUNT(*)
        FROM (
            SELECT pedido_faltante
            FROM electrocasa_dev.silver.devoluciones

            UNION ALL

            SELECT pedido_faltante
            FROM electrocasa_dev.silver.devoluciones_cuarentena
        )
        WHERE pedido_faltante = true
    ) AS pedidos_faltantes;

In [0]:
%sql

SELECT COUNT(*) AS reembolsos_negativos_unicos
FROM (
    SELECT DISTINCT
        devolucion_id,
        pedido_id,
        sucursal_id,
        producto_id,
        motivo,
        monto_reembolso,
        fecha_devolucion
    FROM electrocasa_dev.bronze.devoluciones
)
WHERE CAST(monto_reembolso AS DOUBLE) < 0;